In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
import os

In [19]:
def load_series(filepath):
    df = pd.read_csv(filepath)
    date_col, value_col = df.columns[0], df.columns[1]

    try:
        df[date_col] = pd.to_datetime(df[date_col])
    except Exception:
        df[date_col] = pd.to_datetime(
            df[date_col] + '-1', format='%G-W%V-%u'
        )

    df = df[[date_col, value_col]].set_index(date_col)
    df.columns = ['value']
    return df

def adf_pvalue(series):
    return adfuller(series.dropna())[1]

def transform(series, name):
    if "airline-passengers" in name:
        return np.log(series).diff(12).diff().dropna(), "log + diff(12) + diff(1)"
    if "sales-of-company-x" in name:
        return np.log(series).diff().dropna(), "log + diff(1)"
    if "boston-armed-robberies" in name:
        return series.diff().dropna(), "diff(1)"
    if "air-temperature" in name:
        return series.diff(12).dropna(), "diff(12)"
    if "dowjones" in name.lower():
        return np.log(series).diff().dropna(), "log + diff(1)"
    if "female-births" in name:
        return series.diff(7).dropna(), "diff(7)"
    return series.diff().dropna(), "diff(1)"

In [20]:
files = [
    "monthly-sales-of-company-x-jan-6.csv",
    "monthly-boston-armed-robberies-j.csv",
    "international-airline-passengers.csv",
    "mean-monthly-air-temperature-deg.csv",
    "weekly-closings-of-the-dowjones-.csv",
    "daily-total-female-births-in-cal.csv",
]

In [21]:
rows = []

for fname in files:
    path = fname
    if not os.path.exists(path):
        print(f"Нет файла: {path}")
        continue

    df = load_series(path)
    series = df['value']

    p_before = adf_pvalue(series)
    transformed, method = transform(series, fname)
    p_after = adf_pvalue(transformed)

    rows.append({
        "Ряд": fname,
        "p-value до": round(p_before, 4),
        "Трансформация": method,
        "p-value после": round(p_after, 4),
        "Стационарен": "да" if p_after <= 0.05 else "нет",
    })


results = pd.DataFrame(rows)
print(results.to_string(index=False))

                                 Ряд  p-value до            Трансформация  p-value после Стационарен
monthly-sales-of-company-x-jan-6.csv      0.9889            log + diff(1)         0.0240          да
monthly-boston-armed-robberies-j.csv      0.9943                  diff(1)         0.0000          да
international-airline-passengers.csv      0.9919 log + diff(12) + diff(1)         0.0002          да
mean-monthly-air-temperature-deg.csv      0.0170                 diff(12)         0.0000          да
weekly-closings-of-the-dowjones-.csv      0.6225            log + diff(1)         0.0000          да
daily-total-female-births-in-cal.csv      0.0001                  diff(7)         0.0000          да


/tmp/ipykernel_1651/1803104886.py:17: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  return adfuller(series.dropna())[1]
/tmp/ipykernel_1651/1803104886.py:17: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  return adfuller(series.dropna())[1]
/tmp/ipykernel_1651/1803104886.py:17: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and au